In [1]:
# Установка пакетов (первая ячейка по условиям задания).
# pip запускаем из того же интерпретатора, где работает ядро, -
# иначе Windows может подхватить системный pip и поставить пакеты мимо окружения.
import sys
!{sys.executable} -m pip install -q sentence-transformers scikit-learn pymorphy3 trafilatura pandas


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: C:\Users\ssz\.zcode\workspace\default\lab2\.venv\Scripts\python.exe -m pip install --upgrade pip


# Лабораторная работа 2. Семантическое ядро на эмбеддингах

**Команда:** Данил, Вика, Полина — https://teamdanilvikapolina.github.io/

**Данные:** `queries.csv` — выгрузка Вордстата (тематика «госдума», 2000 запросов, 22.09.2026—22.10.2026).

**Зафиксированные параметры для воспроизводимости:**

- модель эмбеддингов: `intfloat/multilingual-e5-small`, префикс `query:`, `normalize_embeddings=True`;
- рабочий порог агломеративной кластеризации: **0,075** (обоснование в задании 3);
- `random_state`: 42 (K-means, пары, `train_test_split`), 101/202 (перемешивание файлов разметки), 1 (выбор кластеров-примеров), 7 (выборка интентов).

Ноутбук запускается из своей папки: рядом лежат `queries.csv` и файлы разметки (`pairs_A_v1.csv`, `pairs_A_v2.csv`, `pairs_B_v1.csv`, `pairs_B_v2.csv`, `zs_intents.csv`, `intents_v1.csv`).

## Задание 1. Данные: чистка и нормализация

Выгрузка Вордстата: разделитель `;`, переводы строк — голый `\r` (старый Mac-формат), третья колонка — служебный заголовок. Открываем файл сами: universal newlines превращает `\r` в `\n` до pandas.

In [2]:
import os
import re
import pandas as pd
from pymorphy3 import MorphAnalyzer

df = pd.read_csv(
    open("queries.csv", encoding="utf-8-sig"),
    sep=";",
    header=0,
    names=["query", "shows", "extra"],
)
df = df.drop(columns="extra").dropna(subset=["query", "shows"])
df["shows"] = df["shows"].astype(int)
print("Размер:", df.shape)
print(df.head(3).to_string(index=False))

morph = MorphAnalyzer()
cache = {}

def lemma(word):
    if word not in cache:                # кэш обязателен: parse небыстрый
        cache[word] = morph.parse(word)[0].normal_form
    return cache[word]

def normalize(query):
    words = re.findall(r"[а-яёa-z0-9]+", query.lower())
    return " ".join(sorted(lemma(w) for w in words))

df["norm"] = df["query"].apply(normalize)

core = (df.groupby("norm")
          .agg(query=("query", "first"), shows=("shows", "sum"))
          .reset_index()
          .sort_values("shows", ascending=False, ignore_index=True))

print(f"было {len(df)}, после склейки {len(core)} "
      f"(потеряли {len(df) - len(core)} строк, {100 * (len(df) - len(core)) / len(df):.1f}%)")
print("\nТоп-5 запросов по суммарной частотности:")
print(core.head(5)[["query", "shows"]].to_string(index=False))

dup_sizes = df.groupby("norm").size()
print(f"\nГрупп-склеек: {(dup_sizes > 1).sum()}, максимум строк в одной склейке: {dup_sizes.max()}")
for norm in dup_sizes[dup_sizes > 1].sort_values(ascending=False).index[:2]:
    group = df[df["norm"] == norm]
    print(f"\nСклейка [{norm}] -> суммарно {group['shows'].sum()} показов:")
    for _, row in group.iterrows():
        print(f"   «{row['query']}» = {row['shows']}")

Размер: (2000, 2)
           query   shows
         госдума 6647417
выборы в госдуму 4906870
    госдума 2026 4453190
было 2000, после склейки 1975 (потеряли 25 строк, 1.2%)

Топ-5 запросов по суммарной частотности:
                query   shows
              госдума 6647417
     выборы в госдуму 4906870
         госдума 2026 4453190
выборы в госдуму 2026 3993500
 результаты в госдуму 1231511

Групп-склеек: 25, максимум строк в одной склейке: 2

Склейка [2026 в выборы госдума кто на победить] -> суммарно 468922 показов:
   «кто победил на выборах в госдуму 2026» = 234461
   «кто победит на выборах в госдуму 2026» = 234461

Склейка [2026 в выборы госдума на партия программа] -> суммарно 16994 показов:
   «партии на выборах в госдуму 2026 программа» = 8497
   «программы партий на выборах в госдуму 2026» = 8497


**Результат:** было 2000 строк, после склейки 1975 (25 склеек по паре формулировок). Пример склейки для отчёта: «кто победил на выборах в госдуму 2026» и «кто победит на выборах в госдуму 2026» → одна норма, 468 922 показа. Частотности складываем, а не берём максимум: склеенные варианты — один и тот же спрос.

## Задание 2. Бейзлайн: TF-IDF и K-means

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

X = TfidfVectorizer().fit_transform(core["norm"])

k = max(2, len(core) // 20)
labels_tfidf = KMeans(n_clusters=k, n_init=10, random_state=42).fit_predict(X)

score = silhouette_score(X, labels_tfidf, metric="cosine")
print(f"k = {k}, силуэт = {score:.3f}")

core["cluster_tfidf"] = labels_tfidf
for c in core["cluster_tfidf"].drop_duplicates().sample(3, random_state=1):
    print(f"\n--- кластер {c}")
    print(core.loc[core["cluster_tfidf"] == c, "query"].head(5).to_string(index=False))

k = 98, силуэт = 0.273

--- кластер 80
                 госдума саратов
        выборы в госдуму саратов
   выборы в госдуму 2026 саратов
     кандидаты в госдуму саратов
кандидаты в госдуму 2026 саратов

--- кластер 12
                       госдума ярославской области
            выборы в госдуму в ярославской области
         выборы в госдуму 2026 ярославская область
        кандидаты в госдуму от ярославской области
результаты выборов в госдуму в ярославской области

--- кластер 72
                  коммунисты в госдуме
           выборы в госдуму коммунисты
           коммунисты россии в госдуме
      выборы в госдуму 2026 коммунисты
коммунисты россии на выборах в госдуму


**Комментарий к кластерам:** правильно — внутри кластера один объект (регион, партия). Явно нет — `k = len(core) // 20` — навязанная гипотеза: региональные темы дают десятки однотипных кластеров, и собирать из них карту контента нельзя.

## Задание 3. Кластеризация на эмбеддингах

Префикс `query:` — часть протокола e5; без него качество тихо хуже. `normalize_embeddings=True` делает косинусные пороги осмысленными. `linkage="average"` — потому что ward работает только с евклидовой метрикой.

In [4]:
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.cluster import AgglomerativeClustering, HDBSCAN
from sklearn.metrics import silhouette_score
from sklearn.metrics.pairwise import cosine_distances

model = SentenceTransformer("intfloat/multilingual-e5-small")
emb = model.encode([f"query: {q}" for q in core["norm"]],
                   normalize_embeddings=True, batch_size=64,
                   show_progress_bar=True)
print("emb:", emb.shape)

D_full = cosine_distances(emb)
iu = np.triu_indices(len(emb), k=1)
dists = D_full[iu]
print(f"попарные косинусные расстояния: медиана {np.median(dists):.3f}, "
      f"10% квантиль {np.quantile(dists, 0.10):.3f}, "
      f"90% квантиль {np.quantile(dists, 0.90):.3f}")

def clusterize(threshold):
    return AgglomerativeClustering(
        n_clusters=None, distance_threshold=threshold,
        metric="cosine", linkage="average").fit_predict(emb)

def sil(labels, X):
    n = len(set(labels))
    return silhouette_score(X, labels, metric="cosine") if 1 < n < len(labels) else float("nan")

print("\nТаблица: порог -> число кластеров -> силуэт")
for threshold in (0.10, 0.15, 0.20, 0.25):
    labels = clusterize(threshold)
    print(f"порог {threshold:.2f}: кластеров {len(set(labels))}, силуэт {sil(labels, emb):.3f}")

print("\nРасширение сетки вниз (тема узкая, медианное расстояние 0.116):")
for threshold in (0.075, 0.05):
    labels = clusterize(threshold)
    n = len(set(labels))
    single = (pd.Series(labels).value_counts() == 1).sum()
    print(f"порог {threshold:.3f}: кластеров {n}, силуэт {sil(labels, emb):.3f}, "
          f"одноэлементных {single}")

THRESHOLD = 0.075
labels_emb = clusterize(THRESHOLD)
core["cluster_emb"] = labels_emb
sizes = pd.Series(labels_emb).value_counts()
print(f"\nРабочий порог {THRESHOLD}: кластеров {len(set(labels_emb))}, "
      f"силуэт {sil(labels_emb, emb):.3f}, "
      f"размеры: макс {sizes.max()}, медиана {sizes.median():.0f}")

# Дополнительно: HDBSCAN (EOM; cluster_selection_epsilon с precomputed
# ломается на внутреннем баге scikit-learn 1.9.1)
hdb = HDBSCAN(metric="cosine", copy=True)
labels_hdb = hdb.fit_predict(emb)
n_noise = int((labels_hdb == -1).sum())
mask = labels_hdb != -1
n_cl = len(set(labels_hdb[mask]))
print(f"\nHDBSCAN: кластеров {n_cl}, шум {n_noise} ({100*n_noise/len(core):.0f}%), "
      f"силуэт без шума {sil(labels_hdb[mask], emb[mask]):.3f}")
print("Примеры шума (в узком корпусе это редкие одиночные интенты, а не «кривые строки»):")
for q in core.loc[labels_hdb == -1, "query"].sample(min(5, n_noise), random_state=42):
    print("  ", q)

C:\Users\ssz\.zcode\workspace\default\lab2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4935.53it/s]

Batches:   0%|          | 0/31 [00:00<?, ?it/s]

Batches:   3%|▎         | 1/31 [00:00<00:04,  7.10it/s]

...

порог 0.20: кластеров 1, силуэт nan


порог 0.25: кластеров 1, силуэт nan

Расширение сетки вниз (тема узкая, медианное расстояние 0.116):


порог 0.075: кластеров 184, силуэт 0.265, одноэлементных 24


порог 0.050: кластеров 440, силуэт 0.293, одноэлементных 119



Рабочий порог 0.075: кластеров 184, силуэт 0.265, размеры: макс 543, медиана 4

HDBSCAN: кластеров 41, шум 1211 (61%), силуэт без шума 0.277
Примеры шума (в узком корпусе это редкие одиночные интенты, а не «кривые строки»):
   партии кандидаты в госдуму
   исаев госдума
   выборы в госдуму дэг
   результаты в госдуму по округам
   кто проходит в госдуму от единой россии


**Выбранный порог — 0,075.** Пороги 0,15–0,25 из сетки ТЗ дают один кластер на весь корпус («порог слишком велик»): медианное попарное расстояние корпуса — 0,116, тема слишком узкая, поэтому сетка сдвинута вниз по гистограмме попарных расстояний. При 0,10 — два мега-кластера по ~30% корпуса; при 0,05 — 119 одноэлементных кластеров. 0,075: 184 кластера, силуэт 0,265, медиана 4 запроса.

## Задание 4. Честная оценка: силуэт против ручной разметки

Пары внутри кластеров, 50 штук на метод (`random.seed(42)`). Перед разметкой файлы обезличены: нейтральные имена `pairs_A`/`pairs_B` и перемешанный порядок (`random_state` 101/202) — проверяющие не знают, какой файл какому методу принадлежит. Вердикты двух проходов разметки лежат в `pairs_A_v1/v2.csv` и `pairs_B_v1/v2.csv` (критерий: «должны вести на одну страницу», да/нет).

In [5]:
import random

def make_pairs(cluster_col):
    random.seed(42)
    by_cluster = core.groupby(cluster_col)["query"].apply(list)
    candidates = [qs for qs in by_cluster if len(qs) >= 2]
    pairs = set()
    while len(pairs) < 50:
        a, b = random.sample(random.choice(candidates), 2)
        pairs.add(tuple(sorted((a, b))))
    return pd.DataFrame(sorted(pairs), columns=["query_1", "query_2"])

p_emb = make_pairs("cluster_emb")
p_tfi = make_pairs("cluster_tfidf")

a_shuffled = p_emb.sample(frac=1, random_state=101).reset_index(drop=True)  # pairs_A
b_shuffled = p_tfi.sample(frac=1, random_state=202).reset_index(drop=True)  # pairs_B
a_shuffled.to_csv("pairs_A.csv", index=False)
b_shuffled.to_csv("pairs_B.csv", index=False)

def assemble(raw, shuffled, v1, v2):
    out = raw.copy()
    out["verdict_1"] = v1["verdict"].values
    out["verdict_2"] = v2["verdict"].values
    # контроль: порядок обезличенного файла соответствует raw
    left = shuffled.sort_values(["query_1", "query_2"]).reset_index(drop=True)
    right = raw.sort_values(["query_1", "query_2"]).reset_index(drop=True)
    assert (left == right).all().all(), "файл разметки не соответствует парам"
    return out

pairs_emb = assemble(p_emb, a_shuffled,
                     pd.read_csv("pairs_A_v1.csv"), pd.read_csv("pairs_A_v2.csv"))
pairs_tfidf = assemble(p_tfi, b_shuffled,
                       pd.read_csv("pairs_B_v1.csv"), pd.read_csv("pairs_B_v2.csv"))
pairs_emb.to_csv("pairs_emb.csv", index=False)
pairs_tfidf.to_csv("pairs_tfidf.csv", index=False)

def metrics(df, name):
    yes1 = (df["verdict_1"] == "да").mean()
    yes2 = (df["verdict_2"] == "да").mean()
    agree = (df["verdict_1"] == df["verdict_2"]).mean()
    print(f"{name}: доля «да» пр1 = {yes1:.2f}, пр2 = {yes2:.2f}, согласие = {agree:.2f}")
    return {"метод": name, "да1": yes1, "да2": yes2, "согласие": agree}

m_e = metrics(pairs_emb, "Эмбеддинги + порог 0,075")
m_t = metrics(pairs_tfidf, "TF-IDF + K-means")

summary = pd.DataFrame([m_t, m_e]).assign(
    кластеров=[len(set(core["cluster_tfidf"])), len(set(core["cluster_emb"]))],
    силуэт=[silhouette_score(X, core["cluster_tfidf"], metric="cosine"),
            silhouette_score(emb, core["cluster_emb"], metric="cosine")],
)[["метод", "кластеров", "силуэт", "да1", "да2", "согласие"]]
print("\nСводная таблица:")
print(summary.to_string(index=False))

Эмбеддинги + порог 0,075: доля «да» пр1 = 0.58, пр2 = 0.58, согласие = 1.00
TF-IDF + K-means: доля «да» пр1 = 0.74, пр2 = 0.74, согласие = 1.00

Сводная таблица:
                   метод  кластеров   силуэт  да1  да2  согласие
        TF-IDF + K-means         98 0.273024 0.74 0.74       1.0
Эмбеддинги + порог 0,075        184 0.264803 0.58 0.58       1.0


**Вывод:** победил бейзлайн TF-IDF + K-means — выше и силуэт (0,273 против 0,265), и доля «да» (0,74 против 0,58); силуэт и ручная проверка в этом прогоне согласны. Причина: на узком корпусе (все запросы про одну Госдуму) K-means нарезает плотные лексические группы, а эмбеддинги при 0,075 склеивают смысловых соседей с разными интентами («выборы в области» + «госдума области», «коммунисты» + партия «Коммунисты России»). Для турнира это не отмена эмбеддингов: слово «квазибуплон» незнакомо и TF-IDF-словарю, и Вордстату — там работает только семантика (задание 6).

**Оговорка:** разметку выполнял ИИ-ассистент в два прохода (согласие 1,00 — воспроизводимость одного «мозга», а не субъективность людей); файлы `pairs_A.csv`/`pairs_B.csv` обезличены и готовы к перепроверке двумя членами команды.

## Задание 5. Классификация интента

100 случайных запросов (`random_state=7`) размечены классами info/nav/comm/trans. В информационной теме классы сильно перекошены (info — 94 из 100), класс comm встречается 1 раз — `stratify=y` технически невозможен (минимум 2 представителя), поэтому сплит без стратификации, `random_state=42` фиксирует состав.

In [6]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

sample = pd.read_csv("sample_100.csv") if os.path.exists("sample_100.csv") else None
if sample is None:
    sample = pd.DataFrame({"query": core["query"].sample(100, random_state=7).reset_index(drop=True)})
    sample.to_csv("sample_100.csv", index=False)
intents_v1 = pd.read_csv("intents_v1.csv")
marked = sample.copy()
marked["intent"] = intents_v1["intent"].values
marked.to_csv("intents.csv", index=False)

idx = core["query"].isin(marked["query"])
E = emb[idx.values]
order = core.loc[idx, "query"]
ref_by_q = marked.set_index("query")["intent"]
zs_by_q = pd.read_csv("zs_intents.csv").set_index("query")["intent"]
y = ref_by_q.loc[order].values
y_zs = zs_by_q.loc[order].values
print("классы в эталоне:", pd.Series(y).value_counts().to_dict())

E_tr, E_te, y_tr, y_te = train_test_split(E, y, test_size=0.3, random_state=42)
clf = LogisticRegression(max_iter=1000, class_weight="balanced").fit(E_tr, y_tr)
pred = clf.predict(E_te)
print("\n=== Логистическая регрессия на эмбеддингах (тест 30%) ===")
print(classification_report(y_te, pred, zero_division=0))
print(confusion_matrix(y_te, pred, labels=["info", "nav", "comm", "trans"]))

print("\n=== Zero-shot ИИ-ассистент против эталона (все 100) ===")
print(classification_report(y, y_zs, zero_division=0))
print(confusion_matrix(y, y_zs, labels=["info", "nav", "comm", "trans"]))

классы в эталоне: {'info': 94, 'trans': 3, 'nav': 2, 'comm': 1}

=== Логистическая регрессия на эмбеддингах (тест 30%) ===
              precision    recall  f1-score   support

        info       0.96      0.83      0.89        29
         nav       0.00      0.00      0.00         1
       trans       0.00      0.00      0.00         0

    accuracy                           0.80        30
   macro avg       0.32      0.28      0.30        30
weighted avg       0.93      0.80      0.86        30

[[24  4  0  1]
 [ 1  0  0  0]
 [ 0  0  0  0]
 [ 0  0  0  0]]

=== Zero-shot ИИ-ассистент против эталона (все 100) ===
              precision    recall  f1-score   support

        comm       1.00      1.00      1.00         1
        info       0.99      0.98      0.98        94
         nav       1.00      1.00      1.00         2
       trans       0.50      0.67      0.57         3

    accuracy                           0.97       100
   macro avg       0.87      0.91      0.89       10

**Какая пара путается чаще:** info ↔ trans. «результаты выборов онлайн», «избирательные округа», «электронное голосование» — на границе «узнать» и «сделать»: грань проходит по тому, хочет ли пользователь уже сервис, а не статью. Логрегрессия путает info с nav («цик выборы в госдуму 2026» ведёт на конкретный ресурс).

**Выбор метода для ядра на 3000 запросов:** zero-shot ИИ-ассистент — не нужна обучающая разметка (часы работы на 3000 запросов), точность на тесте выше (0,97 против 0,80), перезапуск с фиксированным промптом воспроизводим. Логрегрессия станет лучше, когда размеченных данных станет много; сегодня приоритет — стоимость и скорость.

## Задание 6. Турнир: ядро «квазибуплона» и карта контента

44 формулировки закрывают все направления: определение, расчёт, применение и свойства, сравнения и «зачем», варианты написания (латиница, опечатки, падежи, множественное число). Термину не приписывается никакой реальной истории или научного статуса — по правилам турнира.

In [7]:
kvazi = [
    "что такое квазибуплон", "квазибуплон это", "квазибуплон простыми словами",
    "определение квазибуплона", "квазибуплон определение",
    "значение слова квазибуплон", "квазибуплон что это значит",
    "что означает слово квазибуплон",
    "как считается квазибуплон", "как посчитать квазибуплон",
    "формула квазибуплона", "формула расчета квазибуплона",
    "пример расчета квазибуплона", "пример квазибуплона студента",
    "как вычислить квазибуплон", "калькулятор квазибуплона",
    "расчет квазибуплонов",
    "для чего нужен квазибуплон", "зачем нужен квазибуплон",
    "где применяется квазибуплон", "из чего складывается квазибуплон",
    "от чего зависит квазибуплон", "как повысить квазибуплон",
    "как увеличить квазибуплон студента",
    "может ли квазибуплон быть больше единицы",
    "какой квазибуплон считается хорошим", "шкала квазибуплона",
    "чем квазибуплон отличается от среднего балла",
    "квазибуплон и оценка в чем разница", "что показывает квазибуплон",
    "кому нужен квазибуплон", "плюсы и минусы квазибуплона",
    "стоит ли доверять квазибуплону", "квазибуплон отзывы",
    "kvazibuplon что это", "kvazibuplon formula",
    "что такое квазибуплоно", "что такое гвазибуплон",
    "квазибуплона определение", "квазибуплоном называется",
    "про квазибуплон", "квазибуплон оценка студента",
    "квазибуплон вуз", "квазибуплоны примеры",
]
df_kvazi = pd.DataFrame({"query": kvazi})
print(len(df_kvazi), "формулировок")

# Проблема: pymorphy3 не знает термин и угадывает лемму по-разному
morph_bad = MorphAnalyzer()
for w in ("квазибуплон", "квазибуплоны", "квазибуплонов", "квазибуплона"):
    print(f"без KEEP: {w} -> {morph_bad.parse(w)[0].normal_form}")

# Лечение: список исключений KEEP
KEEP = {"квазибуплон", "квазибуплона", "квазибуплону", "квазибуплоном",
        "квазибуплоне", "квазибуплоны", "квазибуплонов"}

def lemma_keep(word):
    if word in KEEP:
        return "квазибуплон"          # термина нет в словаре, лемму задаём сами
    return lemma(word)

def normalize_keep(query):
    words = re.findall(r"[а-яёa-z0-9]+", query.lower())
    return " ".join(sorted(lemma_keep(w) for w in words))

df_kvazi["norm"] = df_kvazi["query"].apply(normalize_keep)
print("\nс KEEP:", df_kvazi["norm"].nunique(), "уникальных норм из", len(df_kvazi))
dups = df_kvazi[df_kvazi.duplicated("norm", keep=False)].sort_values("norm")
for norm, grp in dups.groupby("norm"):
    print(f"  склейка [{norm}]: {' | '.join(grp['query'])}")

44 формулировок
без KEEP: квазибуплон -> квазибуплоно
без KEEP: квазибуплоны -> квазибуплона
без KEEP: квазибуплонов -> квазибуплоновый
без KEEP: квазибуплона -> квазибуплоно

с KEEP: 42 уникальных норм из 44
  склейка [квазибуплон определение]: определение квазибуплона | квазибуплон определение | квазибуплона определение


In [8]:
emb_kvazi = model.encode([f"query: {q}" for q in df_kvazi["norm"]],
                         normalize_embeddings=True, batch_size=64)
for threshold in (0.075, 0.10, 0.15):
    labels = AgglomerativeClustering(
        n_clusters=None, distance_threshold=threshold,
        metric="cosine", linkage="average").fit_predict(emb_kvazi)
    n = len(set(labels))
    s = sil(labels, emb_kvazi)
    print(f"порог {threshold}: кластеров {n}, силуэт {s:.3f}")

# 0,10 даёт мега-кластер из 39 из 44 запросов — бесполезен для карты контента.
# Рабочий порог тот же, что в задании 3.
THRESHOLD_KVAZI = 0.075
labels_kvazi = AgglomerativeClustering(
    n_clusters=None, distance_threshold=THRESHOLD_KVAZI,
    metric="cosine", linkage="average").fit_predict(emb_kvazi)
df_kvazi["cluster"] = labels_kvazi
df_kvazi.to_csv("kvazi_core.csv", index=False)
print(f"\nрабочий порог {THRESHOLD_KVAZI}: {len(set(labels_kvazi))} кластеров")
for c in sorted(set(labels_kvazi)):
    qs = df_kvazi.loc[df_kvazi["cluster"] == c, "query"].tolist()
    print(f"  кластер {c} (n={len(qs)}): {'; '.join(qs[:6])}{' …' if len(qs) > 6 else ''}")

порог 0.075: кластеров 12, силуэт 0.182
порог 0.1: кластеров 5, силуэт 0.250
порог 0.15: кластеров 1, силуэт nan

рабочий порог 0.075: 12 кластеров
  кластер 0 (n=28): что такое квазибуплон; квазибуплон это; квазибуплон простыми словами; определение квазибуплона; квазибуплон определение; значение слова квазибуплон …
  кластер 1 (n=4): пример квазибуплона студента; квазибуплон оценка студента; квазибуплон вуз; квазибуплоны примеры
  кластер 2 (n=2): чем квазибуплон отличается от среднего балла; квазибуплон и оценка в чем разница
  кластер 3 (n=1): kvazibuplon formula
  кластер 4 (n=2): как повысить квазибуплон; как увеличить квазибуплон студента
  кластер 5 (n=1): стоит ли доверять квазибуплону
  кластер 6 (n=1): что такое гвазибуплон
  кластер 7 (n=1): от чего зависит квазибуплон
  кластер 8 (n=1): может ли квазибуплон быть больше единицы
  кластер 9 (n=1): плюсы и минусы квазибуплона
  кластер 10 (n=1): kvazibuplon что это
  кластер 11 (n=1): где применяется квазибуплон


Страницы сайта (текст извлекается trafilatura из исходного HTML — без JS) кодируются той же моделью с префиксом `query:`; при недоступности сети используется копия страниц из корня репозитория (`../index.html`).

In [9]:
BASE = "https://teamdanilvikapolina.github.io"
from trafilatura import extract

pages = {"/": None, "/chto-takoe-kvazibuplon.html": None}
try:
    from trafilatura import fetch_url
    for path in pages:
        pages[path] = extract(fetch_url(BASE + path), include_comments=False) or ""
    print("страницы получены по сети")
except Exception as e:
    print("trafilatura fetch_url недоступен:", repr(e), "- беру копии из корня репозитория")
    for path, local in (("/", "../index.html"),
                        ("/chto-takoe-kvazibuplon.html", "../chto-takoe-kvazibuplon.html")):
        html = open(local, encoding="utf-8").read()
        pages[path] = extract(html, include_comments=False) or ""

page_emb = model.encode([f"query: {t[:500]}" for t in pages.values()],
                        normalize_embeddings=True)

paths = list(pages)
print("\nСопоставление кластеров со страницами (центроиды):")
page_map = {}
for c in sorted(set(labels_kvazi)):
    centroid = emb_kvazi[labels_kvazi == c].mean(axis=0)
    scores = page_emb @ centroid
    order2 = np.argsort(scores)[::-1]
    page_map[c] = (paths[order2[0]], scores[order2[0]], paths[order2[1]], scores[order2[1]])
    print(f"кластер {c}: {paths[order2[0]]} {scores[order2[0]]:.2f} | "
          f"вторая: {paths[order2[1]]} {scores[order2[1]]:.2f}")

страницы получены по сети

Сопоставление кластеров со страницами (центроиды):
кластер 0: /chto-takoe-kvazibuplon.html 0.88 | вторая: / 0.86
кластер 1: /chto-takoe-kvazibuplon.html 0.89 | вторая: / 0.86
кластер 2: /chto-takoe-kvazibuplon.html 0.87 | вторая: / 0.84
кластер 3: / 0.82 | вторая: /chto-takoe-kvazibuplon.html 0.81
кластер 4: /chto-takoe-kvazibuplon.html 0.86 | вторая: / 0.86
кластер 5: /chto-takoe-kvazibuplon.html 0.85 | вторая: / 0.83
кластер 6: /chto-takoe-kvazibuplon.html 0.85 | вторая: / 0.84
кластер 7: /chto-takoe-kvazibuplon.html 0.86 | вторая: / 0.85
кластер 8: /chto-takoe-kvazibuplon.html 0.84 | вторая: / 0.82
кластер 9: /chto-takoe-kvazibuplon.html 0.84 | вторая: / 0.84
кластер 10: / 0.87 | вторая: /chto-takoe-kvazibuplon.html 0.87
кластер 11: / 0.83 | вторая: /chto-takoe-kvazibuplon.html 0.83


**Карта контента** (полная таблица с решениями — в `report-lab2.md`):

- определение/расчёт/примеры/отличия/FAQ-темы — закрывает статья (0,84–0,89): ок;
- латиница — тянет к главной (в её title есть `kvazibuplon`): ок;
- **пробел — кластеры «как повысить» (0,86/0,86), «плюсы и минусы» (0,84/0,84), «где применяется» (0,83/0,83)**: близости размазаны между статьёй и главной, ни одна страница теме не отвечает;
- лёгкая каннибализация «kvazibuplon что это» (0,87/0,87) — оставляем: главная навигационная, отдавать ей латиницу нормально.

**Ровно одно изменение:** страница `/kak-povysit-kvazibuplon.html` под крупнейший пробел (кластеры «как повысить / где применяется / плюсы и минусы»), с разметкой DefinedTerm + FAQPage, обновлённым `sitemap.xml` и отправкой через IndexNow. Страницу «как считается» не создаём — расчёт уже закрыт статьёй (0,88), была бы каннибализация.